# Presentation Videos — 4 Packs

In [1]:
import sys, os, re

PROJ = '/groups/jingyiliu/home/liuj4/cell-gnn'
sys.path.insert(0, PROJ)
os.chdir(PROJ)

import torch
import numpy as np
import zarr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess, shutil

from cell_gnn.config import CellGNNConfig
from cell_gnn.utils import to_numpy
from cell_gnn.models.Siren_Network import Siren
from cell_gnn.models.MLP import MLP
from cell_gnn.graph_utils import scatter_aggregate
from cell_gnn.plot import build_edge_features, _batched_mlp_eval

PROJ = Path(PROJ)
VID_DIR = PROJ / 'presentation' / 'videos'
VID_DIR.mkdir(parents=True, exist_ok=True)

def load_config(n):
    return CellGNNConfig.from_yaml(str(PROJ / 'config' / 'misc' / f'{n}.yaml'))
def load_pos(d):
    return np.array(zarr.open(str(PROJ/'graphs_data'/'misc'/d/'x_list_0'/'pos.zarr'),'r')[:])
def load_field(d):
    return np.array(zarr.open(str(PROJ/'graphs_data'/'misc'/d/'x_list_0'/'field.zarr'),'r')[:])
def load_vel(d):
    return np.array(zarr.open(str(PROJ/'graphs_data'/'misc'/d/'x_list_0'/'vel.zarr'),'r')[:])

def load_checkpoint(config_name):
    md = PROJ / 'log' / 'misc' / config_name / 'models'
    ckpts = sorted(md.glob('best_model_with_0_graphs_0_*.pt'),
                   key=lambda p: int(p.stem.split('_')[-1]))
    if not ckpts:
        ckpts = list(md.glob('best_model_with_0_graphs_0.pt'))
    raw = torch.load(ckpts[-1], map_location='cpu', weights_only=False)
    state = raw.get('model_state_dict', raw) if isinstance(raw, dict) else raw
    return {k.replace('_orig_mod.', ''): v for k, v in state.items()}

def all_checkpoints(config_name):
    """Find ALL checkpoints across all epochs, sorted by (epoch, iter).
    Returns list of (label_str, path)."""
    md = PROJ / 'log' / 'misc' / config_name / 'models'
    pat = re.compile(r'best_model_with_0_graphs_(\d+)_(\d+)\.pt$')
    results = []
    for p in md.glob('best_model_with_0_graphs_*_*.pt'):
        m = pat.search(str(p))
        if m:
            epoch, iteration = int(m.group(1)), int(m.group(2))
            results.append((epoch, iteration, p))
    results.sort(key=lambda x: (x[0], x[1]))
    return [(f'Ep {ep} / {it:,}', p) for ep, it, p in results]

def get_particle_radius(config_name):
    """Get display particle radius from config. r0 = 2 * radius, scaled down for clarity."""
    cfg = load_config(config_name)
    r0 = cfg.simulation.cell_params[0][1]  # [k_rep, r0, kadh, r_on, delta, mu_f]
    return r0 * 0.3  # ~0.015 for r0=0.05

def data_to_scatter_size(radius, fig_width_inches, dpi, data_range=1.0, axis_fraction=0.65):
    """Convert physical particle radius to matplotlib scatter s (points²).
    
    For 3D plots the axis occupies ~65% of the figure width.
    scatter s = pi * (radius_in_points)^2
    """
    fig_width_pts = fig_width_inches * 72  # 1 inch = 72 points
    axis_width_pts = fig_width_pts * axis_fraction
    pts_per_data = axis_width_pts / data_range
    radius_pts = radius * pts_per_data
    return np.pi * radius_pts ** 2

def frames_to_mp4(frame_dir, output_path, fps=30):
    subprocess.run(['ffmpeg','-y','-loglevel','error','-framerate',str(fps),
        '-i',f'{frame_dir}/frame_%06d.png',
        '-vf','scale=trunc(iw/2)*2:trunc(ih/2)*2',
        '-c:v','libx264','-crf','23','-pix_fmt','yuv420p',
        str(output_path)], check=True)
    print(f'  -> {output_path.name}')

def clean_3d_ax(ax):
    """Remove grid, make clean equal-length 3D box."""
    ax.grid(False)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('k')
    ax.yaxis.pane.set_edgecolor('k')
    ax.zaxis.pane.set_edgecolor('k')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_zlim(0, 1)
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

print('Setup done.')

Setup done.


---
## Video functions

In [2]:
def make_simulation_3d(dataset_name, config_name, title, output_name,
                       frame_step=50, fps=30, max_frames=160, has_field=True):
    """3D rotating scatter of GT cell positions with physically correct particle size."""
    pos = load_pos(dataset_name)
    field = load_field(dataset_name)
    T = pos.shape[0]
    idxs = np.arange(0, min(T, max_frames * frame_step), frame_step)
    
    # Particle size from config: r0 = 2 * radius
    particle_r = get_particle_radius(config_name)
    fig_w = 7
    dpi = 120
    s = data_to_scatter_size(particle_r, fig_w, dpi)
    print(f'  particle radius = {particle_r}, scatter s = {s:.1f}')
    
    # Check if field has any nonzero values
    f_all = field[idxs, :, 0]
    nz = f_all[f_all != 0]
    use_field = has_field and len(nz) > 0
    if use_field:
        vmin, vmax = np.percentile(nz, [2, 98])
        if vmin == vmax: vmin, vmax = -1, 1
    
    tmp = VID_DIR / f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    fig = plt.figure(figsize=(fig_w, fig_w))
    ax = fig.add_subplot(111, projection='3d')
    
    for i, fi in enumerate(idxs):
        ax.clear()
        p = pos[fi]
        if use_field:
            f = field[fi, :, 0]
            ax.scatter(p[:,0], p[:,1], p[:,2], c=f, s=s, alpha=0.7,
                       cmap='coolwarm', vmin=vmin, vmax=vmax,
                       edgecolors='none', depthshade=True)
        else:
            ax.scatter(p[:,0], p[:,1], p[:,2], s=s, alpha=0.7,
                       c='#1f77b4', edgecolors='none', depthshade=True)
        clean_3d_ax(ax)
        ax.set_title(f'{title}\n$t = {fi*0.002:.2f}$', fontsize=12)
        ax.view_init(elev=25, azim=30 + i * 0.5)
        fig.savefig(tmp / f'frame_{i:06d}.png', dpi=dpi, bbox_inches='tight')
        if i % 50 == 0: print(f'  {i}/{len(idxs)}')
    plt.close(fig)
    frames_to_mp4(tmp, VID_DIR / f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

In [3]:
def make_reconstruction_3d(dataset_name, config_name, title, output_name,
                           frame_step=50, fps=20, max_frames=160, n_rollout=10):
    """Side-by-side 3D: GT positions (left) vs model rollout positions (right)."""
    pos_all = load_pos(dataset_name)
    T, N, _ = pos_all.shape
    dt = 0.002
    
    cfg = load_config(config_name)
    dim = cfg.simulation.dimension
    emb_dim = cfg.graph_model.embedding_dim
    max_radius = cfg.simulation.max_radius
    isz = dim + 1 + emb_dim
    
    # Particle size from config
    particle_r = get_particle_radius(config_name)
    fig_w = 14  # side-by-side, each panel ~7 inches
    dpi = 120
    s = data_to_scatter_size(particle_r, fig_w / 2, dpi)
    print(f'  particle radius = {particle_r}, scatter s = {s:.1f}')
    
    state = load_checkpoint(config_name)
    lin_edge = MLP(input_size=isz, output_size=cfg.graph_model.output_size,
                   nlayers=cfg.graph_model.n_layers, hidden_size=cfg.graph_model.hidden_dim,
                   device='cpu')
    es = {k.replace('lin_edge.',''):v for k,v in state.items() if k.startswith('lin_edge.')}
    lin_edge.load_state_dict(es); lin_edge.eval()
    med_emb = state['a'][0].median(dim=0).values if 'a' in state else torch.zeros(emb_dim)
    
    def predict_vel(pos_t):
        N = pos_t.shape[0]
        dist = torch.cdist(pos_t, pos_t)
        pred = torch.zeros(N, 3)
        for ci in range(N):
            nbrs = (dist[ci] < max_radius) & (dist[ci] > 0)
            if not nbrs.any(): continue
            dp = (pos_t[nbrs] - pos_t[ci]) / max_radius
            r = dist[ci, nbrs].unsqueeze(1) / max_radius
            emb = med_emb.unsqueeze(0).expand(dp.shape[0], -1)
            inp = torch.cat([dp, r, emb], dim=-1)
            pred[ci] = lin_edge(inp).sum(dim=0)
        return pred
    
    idxs = np.arange(0, min(T - n_rollout, max_frames * frame_step), frame_step)
    tmp = VID_DIR / f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    fig = plt.figure(figsize=(fig_w, 6))
    ax1 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122, projection='3d')
    
    for i, fi in enumerate(idxs):
        with torch.no_grad():
            cur = torch.tensor(pos_all[fi], dtype=torch.float32)
            for _ in range(n_rollout):
                vel = predict_vel(cur)
                cur = (cur + vel * dt) % 1.0
        pred_pos = cur.numpy()
        gt_pos = pos_all[fi + n_rollout]
        
        ax1.clear(); ax2.clear()
        
        ax1.scatter(gt_pos[:,0], gt_pos[:,1], gt_pos[:,2],
                   s=s, alpha=0.7, c='#1f77b4', edgecolors='none', depthshade=True)
        clean_3d_ax(ax1)
        ax1.set_title('Ground Truth', fontsize=12)
        ax1.view_init(elev=25, azim=30 + i*0.5)
        
        ax2.scatter(pred_pos[:,0], pred_pos[:,1], pred_pos[:,2],
                   s=s, alpha=0.7, c='#d62728', edgecolors='none', depthshade=True)
        clean_3d_ax(ax2)
        ax2.set_title(f'Reconstructed ({n_rollout}-step rollout)', fontsize=12)
        ax2.view_init(elev=25, azim=30 + i*0.5)
        
        fig.suptitle(f'{title}   $t = {fi*dt:.2f}$', fontsize=13)
        fig.tight_layout()
        fig.savefig(tmp / f'frame_{i:06d}.png', dpi=dpi, bbox_inches='tight')
        if i % 10 == 0: print(f'  {i}/{len(idxs)}')
    
    plt.close(fig)
    frames_to_mp4(tmp, VID_DIR / f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

In [4]:
def make_force_learning_video(config_name, output_name, fps=8):
    """Stitch existing MLP1 PNGs from log dir into a video.
    
    Uses the actual plots generated during training — no re-rendering needed.
    Covers all epochs. Subsamples to ~30 frames for a 3-4 sec video.
    """
    mlp1_dir = PROJ / 'log' / 'misc' / config_name / 'tmp_training' / 'function' / 'MLP1'
    if not mlp1_dir.exists():
        print(f'  No MLP1 dir: {mlp1_dir}')
        return
    
    # Collect all function_{epoch}_{iter}.png, sorted by (epoch, iter)
    pat = re.compile(r'function_(\d+)_(\d+)\.png$')
    pngs = []
    for p in mlp1_dir.glob('function_*.png'):
        m = pat.search(p.name)
        if m:
            pngs.append((int(m.group(1)), int(m.group(2)), p))
    pngs.sort(key=lambda x: (x[0], x[1]))
    
    if not pngs:
        print(f'  No MLP1 PNGs found'); return
    print(f'  Found {len(pngs)} MLP1 plots')
    
    # Subsample to ~30 frames
    target_frames = 30
    pngs = pngs[::max(1, len(pngs) // target_frames)]
    print(f'  Using {len(pngs)} frames')
    
    # Symlink into tmp dir with sequential naming for ffmpeg
    tmp = VID_DIR / f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    for i, (ep, it, src) in enumerate(pngs):
        dst = tmp / f'frame_{i:06d}.png'
        shutil.copy2(src, dst)
    
    frames_to_mp4(tmp, VID_DIR / f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

In [5]:
def make_field_learning_video(config_name, output_name, resolution=60,
                               frame_step=50, fps=30, max_frames=160):
    """Left: top-down learned field z-slice. Right: 3D particles colored by learned c."""
    cfg = load_config(config_name)
    dim = cfg.simulation.dimension
    n_frames = cfg.simulation.n_frames
    omega = getattr(cfg.graph_model, 'omega_field', 30.0)
    
    # Particle size from config
    particle_r = get_particle_radius(config_name)
    fig_w = 13
    dpi = 120
    s = data_to_scatter_size(particle_r, fig_w / 2, dpi)
    
    state = load_checkpoint(config_name)
    sk = [k for k in state if k.startswith('siren_field.net.')]
    if not sk: print(f'  No siren_field in {config_name}'); return
    nm = len(set(k.split('.')[2] for k in sk))
    hd = state['siren_field.net.0.linear.weight'].shape[0]
    
    last_layer = max(int(k.split('.')[2]) for k in sk)
    out_features = state[f'siren_field.net.{last_layer}.bias'].shape[0]
    
    siren = Siren(in_features=dim+1, out_features=out_features, hidden_features=hd,
                  hidden_layers=max(nm-2,0), outermost_linear=True,
                  first_omega_0=omega, hidden_omega_0=omega)
    ss = {k.replace('siren_field.',''):v for k,v in state.items() if k.startswith('siren_field.')}
    siren.load_state_dict(ss); siren.eval()
    
    c1d = torch.linspace(0,1,resolution)
    gy,gx = torch.meshgrid(c1d,c1d,indexing='ij')
    g2d = torch.stack([gx.reshape(-1),gy.reshape(-1)],dim=1)
    g3d = torch.cat([g2d, 0.5*torch.ones(resolution**2,1)], dim=1)
    
    pos_all = load_pos(cfg.dataset); T = pos_all.shape[0]
    
    vs = []
    with torch.no_grad():
        for tf in np.linspace(0.01,0.99,10):
            out = siren(torch.cat([g3d,tf*torch.ones(resolution**2,1)],1))
            vs.append(out[:,0].numpy())
    av = np.concatenate(vs)
    vmin,vmax = np.percentile(av,[2,98])
    
    tmp = VID_DIR/f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    fig = plt.figure(figsize=(fig_w, 5.5))
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(122, projection='3d')
    fc = 0
    
    for fi in range(0, min(T, max_frames*frame_step), frame_step):
        tf = fi / n_frames
        with torch.no_grad():
            out = siren(torch.cat([g3d,tf*torch.ones(resolution**2,1)],1))
            c = out[:,0].reshape(resolution,resolution).numpy()
        
        ax1.clear(); ax2.clear()
        ax1.imshow(c, origin='lower', extent=[0,1,0,1], cmap='viridis', vmin=vmin, vmax=vmax)
        ax1.set_title('Learned $c(\\mathbf{x},t)$  ($z\\!=\\!0.5$)', fontsize=12)
        ax1.set_xlabel('x'); ax1.set_ylabel('y')
        
        pn = pos_all[min(fi,T-1)]
        with torch.no_grad():
            pt = torch.tensor(pn, dtype=torch.float32)
            out_cell = siren(torch.cat([pt, tf*torch.ones(pt.shape[0],1)],1))
            cc = out_cell[:,0].numpy()
        ax2.scatter(pn[:,0],pn[:,1],pn[:,2], c=cc, s=s, alpha=0.7,
                   cmap='viridis', vmin=vmin, vmax=vmax, edgecolors='none', depthshade=True)
        clean_3d_ax(ax2)
        ax2.set_title('3D cells (color = learned $c$)', fontsize=12)
        ax2.view_init(elev=25, azim=30+fc*0.5)
        
        fig.suptitle(f'Learned Field   $t={fi*0.002:.2f}$', fontsize=13)
        fig.tight_layout()
        fig.savefig(tmp/f'frame_{fc:06d}.png', dpi=dpi, bbox_inches='tight')
        fc += 1
        if fc%50==0: print(f'  {fc}')
    
    plt.close(fig)
    frames_to_mp4(tmp, VID_DIR/f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

---
## Pack 1: Pairwise Only

In [6]:
C1 = 'dicty_spring_force_rk4'
D1 = 'dicty_spring_force_rk4'

print('Pack 1 — Simulation')
make_simulation_3d(D1, C1, 'Pairwise Only', 'pack1_sim', has_field=False)
print('Pack 1 — Force learning')
make_force_learning_video(C1, 'pack1_force')
print('Pack 1 — Reconstruction')
make_reconstruction_3d(D1, C1, 'Pairwise Only', 'pack1_recon', n_rollout=5)

Pack 1 — Simulation
  particle radius = 0.015, scatter s = 75.9
  0/160
  50/160
  100/160
  150/160
  -> pack1_sim.mp4
Pack 1 — Force learning
  Found 80 MLP1 plots
  Using 40 frames
  -> pack1_force.mp4
Pack 1 — Reconstruction
  particle radius = 0.015, scatter s = 75.9
  0/160
  10/160
  20/160
  30/160
  40/160
  50/160
  60/160
  70/160
  80/160
  90/160
  100/160
  110/160
  120/160
  130/160
  140/160
  150/160
  -> pack1_recon.mp4


---
## Pack 2: Pairwise + Static Gaussian Field

In [7]:
C2 = 'dicty_spring_force_rk4_static_field_wide'
D2 = 'dicty_spring_force_rk4_static_field_wide'

if (PROJ/'graphs_data'/'misc'/D2/'x_list_0'/'pos.zarr').exists():
    print('Pack 2 — Simulation')
    make_simulation_3d(D2, C2, 'Pairwise + Static Gaussian', 'pack2_sim')
    if len(all_checkpoints(C2)) > 0:
        print('Pack 2 — Force learning')
        make_force_learning_video(C2, 'pack2_force')
        print('Pack 2 — Field learning')
        make_field_learning_video(C2, 'pack2_field')
        print('Pack 2 — Reconstruction')
        make_reconstruction_3d(D2, C2, 'Static Gaussian', 'pack2_recon', n_rollout=5)
    else:
        print('  Model not trained yet')
else:
    print('  Data not generated yet')

Pack 2 — Simulation
  particle radius = 0.015, scatter s = 75.9
  0/160
  50/160
  100/160
  150/160
  -> pack2_sim.mp4
Pack 2 — Force learning
  Found 400 MLP1 plots
  Using 31 frames
  -> pack2_force.mp4
Pack 2 — Field learning
  50
  100
  150
  -> pack2_field.mp4
Pack 2 — Reconstruction
  particle radius = 0.015, scatter s = 75.9
  0/160
  10/160
  20/160
  30/160
  40/160
  50/160
  60/160
  70/160
  80/160
  90/160
  100/160
  110/160
  120/160
  130/160
  140/160
  150/160
  -> pack2_recon.mp4


---
## Pack 3: Pairwise + Moving Gaussian Field(s)

In [8]:
C3 = 'dicty_spring_force_rk4_dynamic_field_siren_v9'
D3 = 'dicty_spring_force_rk4_dynamic_field_siren_v9'

print('Pack 3 — Simulation (8 moving Gaussians)')
make_simulation_3d(D3, C3, 'Pairwise + Moving Gaussians', 'pack3_sim')
print('Pack 3 — Force learning')
make_force_learning_video(C3, 'pack3_force')
print('Pack 3 — Reconstruction')
make_reconstruction_3d(D3, C3, 'Moving Gaussians', 'pack3_recon', n_rollout=5)
print('Pack 3 — Field: v9 has no SIREN branch, skip')

Pack 3 — Simulation (8 moving Gaussians)
  particle radius = 0.015, scatter s = 75.9
  0/160
  50/160
  100/160
  150/160
  -> pack3_sim.mp4
Pack 3 — Force learning
  Found 20 MLP1 plots
  Using 20 frames
  -> pack3_force.mp4
Pack 3 — Reconstruction
  particle radius = 0.015, scatter s = 75.9
  0/160
  10/160
  20/160
  30/160
  40/160
  50/160
  60/160
  70/160
  80/160
  90/160
  100/160
  110/160
  120/160
  130/160
  140/160
  150/160
  -> pack3_recon.mp4
Pack 3 — Field: v9 has no SIREN branch, skip


---
## Pack 4: Pairwise + Self-Generated Diffusion Field

In [9]:
C4 = 'dicty_spring_force_rk4_diffusion_field_siren_grad_v4'
D4 = 'dicty_spring_force_rk4_diffusion_field_siren_grad_v4'

print('Pack 4 — Simulation (agent-sourced field)')
make_simulation_3d(D4, C4, 'Pairwise + Self-Generated Field', 'pack4_sim')
print('Pack 4 — Force learning')
make_force_learning_video(C4, 'pack4_force')
print('Pack 4 — Field learning')
make_field_learning_video(C4, 'pack4_field')
print('Pack 4 — Reconstruction')
make_reconstruction_3d(D4, C4, 'Self-Generated Field', 'pack4_recon', n_rollout=5)

Pack 4 — Simulation (agent-sourced field)
  particle radius = 0.015, scatter s = 75.9
  0/160
  50/160
  100/160
  150/160
  -> pack4_sim.mp4
Pack 4 — Force learning
  Found 200 MLP1 plots
  Using 34 frames
  -> pack4_force.mp4
Pack 4 — Field learning
  50
  100
  150
  -> pack4_field.mp4
Pack 4 — Reconstruction
  particle radius = 0.015, scatter s = 75.9
  0/160
  10/160
  20/160
  30/160
  40/160
  50/160
  60/160
  70/160
  80/160
  90/160
  100/160
  110/160
  120/160
  130/160
  140/160
  150/160
  -> pack4_recon.mp4
